# PTCG Alakazam v22 — BC Warmup Training

Behavior cloning from v22 self-play (547,796 decisions, 2,000 games). See `docs/nn-training.md` in the repo for the full plan. This notebook:

1. Writes out `encode.py` / `model.py` / `dataset.py` / `train_bc.py` (kept in sync with `training/nn/` in the repo — smoke-tested locally on CPU before this notebook was built).
2. Trains the BC policy/value net (10 epochs, LR 1e-4).
3. Reports held-out top-1 accuracy against v22's chosen actions (the direct imitation-quality metric) and saves the checkpoint.

**Data:** attach dataset `jander6364/ptcg-alakazam-v22-bc-data` as input.
**Accelerator:** GPU T4 x1 (Settings → Accelerator).


In [ ]:
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
!echo '--- /kaggle/input ---'; ls -la /kaggle/input/ 2>&1
!echo '--- recursive (depth 3) ---'; find /kaggle/input -maxdepth 3 2>&1


In [ ]:
%%writefile encode.py
"""Observation -> tensor encoding for the BC/self-play net.

v1 rebuilt design (all prior encoding code was lost with the reset — see
docs/nn-training.md). Deliberately simpler than the original 22000-vocab
transformer-decoder sketch: a 13-slot board sequence + hand/discard bag
embeddings + a small numeric feature vector, and a per-candidate-action
feature vector for the policy head. No cg-lib dependency — everything is
read directly off the raw obs_dict, identical to what `main.py` consumes.

CARD_VOCAB / ATTACK_VOCAB are hardcoded upper bounds (real max observed card
ID is 1267 as of 2026-07-01; enums may grow during the competition per the
official docs, hence the safety margin) rather than calling all_card_data() —
this keeps encode.py usable with or without cg-lib attached.
"""
import math

CARD_VOCAB = 2000
ATTACK_VOCAB = 2000
OPTION_TYPE_VOCAB = 17  # OptionType enum, 0..16
N_BOARD_SLOTS = 13      # my_active, my_bench x5, opp_active, opp_bench x5, stadium
MAX_ACTIONS = 64
NUM_FEATS = 13


def _pk_id(pk):
    return (pk or {}).get("id", 0) or 0


def _active(p):
    a = (p or {}).get("active")
    return a[0] if a and len(a) > 0 and a[0] else None


def board_slot_ids(obs):
    """13 card-id tokens: [my_active, my_bench(5), opp_active, opp_bench(5), stadium]."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    opp = pl[1 - me_idx] if len(pl) == 2 else {}
    my_bench = (me.get("bench") or [])[:5]
    opp_bench = (opp.get("bench") or [])[:5]
    ids = [_pk_id(_active(me))]
    ids += [_pk_id(b) for b in my_bench] + [0] * (5 - len(my_bench))
    ids.append(_pk_id(_active(opp)))
    ids += [_pk_id(b) for b in opp_bench] + [0] * (5 - len(opp_bench))
    stadium = (cur.get("stadium") or [None])
    ids.append(_pk_id(stadium[0]) if stadium else 0)
    return [min(i, CARD_VOCAB - 1) for i in ids]


def hand_ids(obs, cap=20):
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    hand = me.get("hand") or []
    ids = [min(_pk_id(c), CARD_VOCAB - 1) for c in hand if _pk_id(c)]
    return ids[:cap] or [0]


def discard_ids(obs, cap=20):
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    disc = me.get("discard") or []
    ids = [min(_pk_id(c), CARD_VOCAB - 1) for c in disc if _pk_id(c)]
    return ids[:cap] or [0]


def numeric_feats(obs):
    """Fixed-size float feature vector, roughly matching main.py's _census inputs."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    opp = pl[1 - me_idx] if len(pl) == 2 else {}
    my_active = _active(me)
    opp_active = _active(opp)
    my_hp = (my_active or {}).get("hp", 0) or 0
    my_maxhp = (my_active or {}).get("maxHp", 1) or 1
    opp_hp = (opp_active or {}).get("hp", 0) or 0
    opp_maxhp = (opp_active or {}).get("maxHp", 1) or 1
    my_hand_n = me.get("handCount") or len(me.get("hand") or [])
    opp_hand_n = opp.get("handCount", 0) or 0
    my_deck = me.get("deckCount", 0) or 0
    opp_deck = opp.get("deckCount", 0) or 0
    my_prizes = len(me.get("prize") or [])
    opp_prizes = len(opp.get("prize") or [])
    opp_energies = set()
    for ec in (opp_active or {}).get("energyCards") or []:
        opp_energies.add(ec.get("id"))
    opp_mist = 1.0 if (11 in opp_energies or 20 in opp_energies) else 0.0
    return [
        my_hp / max(my_maxhp, 1),
        opp_hp / max(opp_maxhp, 1),
        min(my_hand_n, 30) / 30.0,
        min(opp_hand_n, 30) / 30.0,
        min(my_deck, 60) / 60.0,
        min(opp_deck, 60) / 60.0,
        min(my_prizes, 6) / 6.0,
        min(opp_prizes, 6) / 6.0,
        opp_mist,
        1.0 if cur.get("supporterPlayed") else 0.0,
        1.0 if cur.get("energyAttached") else 0.0,
        1.0 if cur.get("retreated") else 0.0,
        min(cur.get("turn", 0) or 0, 60) / 60.0,
    ]


def _opt_card_id(o, hand, bench):
    """Mirrors main.py._opt_card_id — resolve an option's associated card id."""
    ot = o.get("type")
    idx = o.get("index")
    if ot in (3, 4, 5, 7, 8, 9):  # CARD/TOOL_CARD/ENERGY_CARD/PLAY/ATTACH/EVOLVE
        if idx is not None and 0 <= idx < len(hand):
            return _pk_id(hand[idx])
        return 0
    if ot == 10:  # ABILITY
        area = o.get("area")
        if area == 4:
            return 0  # active resolved separately by caller if needed
        if area == 5 and 0 <= idx < len(bench):
            return _pk_id(bench[idx])
    return 0


def encode_action(obs, o):
    """Per-candidate-option feature dict: type id, resolved card id, attack id, numerics."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    hand = me.get("hand") or []
    bench = me.get("bench") or []
    ot = o.get("type") or 0
    cid = _opt_card_id(o, hand, bench)
    if ot == 10 and o.get("area") == 4:  # ABILITY on active
        cid = _pk_id(_active(me))
    attack_id = o.get("attackId") or 0
    area = o.get("area") or 0
    in_play_area = o.get("inPlayArea") or 0
    index = o.get("index") or 0
    in_play_index = o.get("inPlayIndex") or 0
    return {
        "type": min(ot, OPTION_TYPE_VOCAB - 1),
        "card_id": min(cid, CARD_VOCAB - 1),
        "attack_id": min(attack_id, ATTACK_VOCAB - 1),
        "numeric": [area / 12.0, in_play_area / 12.0, index / 20.0, in_play_index / 6.0],
    }


def encode_sample(obs, sel):
    """Full encoding for one decision point. Returns a dict of plain python
    lists/ints (torch-free) so this module works without a torch import —
    the Dataset class converts to tensors at collate time."""
    opts = (sel.get("option") or [])[:MAX_ACTIONS]
    return {
        "board_ids": board_slot_ids(obs),
        "hand_ids": hand_ids(obs),
        "discard_ids": discard_ids(obs),
        "numeric": numeric_feats(obs),
        "actions": [encode_action(obs, o) for o in opts],
        "n_actions": len(opts),
    }


In [ ]:
%%writefile model.py
"""BC/self-play actor-critic net. See encode.py for the feature design and
docs/nn-training.md for the architecture rationale. Small on purpose: this is
a warm-start policy, not the final word — self-play (Phase 1+) can grow it.
"""
import torch
import torch.nn as nn

from encode import (
    CARD_VOCAB, ATTACK_VOCAB, OPTION_TYPE_VOCAB, N_BOARD_SLOTS, NUM_FEATS,
)

D_CARD = 128
D_ATTACK = 64
D_TYPE = 32
D_MODEL = 128


class PTCGNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.card_embed = nn.Embedding(CARD_VOCAB, D_CARD, padding_idx=0)
        self.attack_embed = nn.Embedding(ATTACK_VOCAB, D_ATTACK, padding_idx=0)
        self.type_embed = nn.Embedding(OPTION_TYPE_VOCAB, D_TYPE)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_CARD, nhead=2, dim_feedforward=256, batch_first=True)
        self.board_transformer = nn.TransformerEncoder(enc_layer, num_layers=1)

        self.hand_bag = nn.EmbeddingBag(CARD_VOCAB, D_CARD, mode="sum", padding_idx=0)
        self.discard_bag = nn.EmbeddingBag(CARD_VOCAB, D_CARD, mode="sum", padding_idx=0)

        self.numeric_proj = nn.Sequential(nn.Linear(NUM_FEATS, D_CARD), nn.ReLU())
        self.trunk = nn.Sequential(
            nn.Linear(D_CARD * 4, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.value_head = nn.Linear(256, 1)

        act_in_dim = D_TYPE + D_CARD + D_ATTACK + 4
        self.action_mlp = nn.Sequential(nn.Linear(act_in_dim, D_MODEL), nn.ReLU())
        self.logit_mlp = nn.Sequential(
            nn.Linear(D_MODEL + 256, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, board_ids, hand_ids, discard_ids, numeric,
                action_type, action_card, action_attack, action_numeric, action_mask):
        board_emb = self.card_embed(board_ids)                  # (B,13,128)
        board_ctx = self.board_transformer(board_emb)            # (B,13,128)
        board_vec = board_ctx.mean(dim=1)                        # (B,128)
        hand_vec = self.hand_bag(hand_ids)                       # (B,128)
        discard_vec = self.discard_bag(discard_ids)              # (B,128)
        feat_vec = self.numeric_proj(numeric)                    # (B,128)

        trunk_in = torch.cat([board_vec, hand_vec, discard_vec, feat_vec], dim=-1)
        trunk = self.trunk(trunk_in)                             # (B,256)
        value = torch.tanh(self.value_head(trunk).squeeze(-1))   # (B,)

        B, A = action_type.shape
        type_emb = self.type_embed(action_type)                  # (B,A,32)
        card_emb = self.card_embed(action_card)                  # (B,A,128)
        attack_emb = self.attack_embed(action_attack)             # (B,A,64)
        act_in = torch.cat([type_emb, card_emb, attack_emb, action_numeric], dim=-1)
        act_vec = self.action_mlp(act_in)                         # (B,A,128)

        trunk_exp = trunk.unsqueeze(1).expand(-1, A, -1)          # (B,A,256)
        logits = self.logit_mlp(torch.cat([act_vec, trunk_exp], dim=-1)).squeeze(-1)  # (B,A)
        logits = logits.masked_fill(action_mask == 0, -1e9)
        return logits, value


In [ ]:
%%writefile dataset.py
"""Turns training/bc_data*.pkl(.gz) shards into batched tensors for PTCGNet.

BC value target = game outcome (+1/-1/0), the simplest valid target for the
imitation warmup. n-step bootstrapped value targets (docs/nn-training.md
§Value Targets) apply to the self-play phase, not this warmup.
"""
import glob
import gzip
import pickle

import torch
from torch.utils.data import Dataset

from encode import encode_sample, MAX_ACTIONS


def _opener(path):
    return gzip.open if path.endswith(".gz") else open


def load_shards(pattern):
    """pattern e.g. '/kaggle/input/**/bc_data*.pkl' (recursive — Kaggle's exact
    mount subdirectory name can differ from the dataset slug)."""
    paths = sorted(glob.glob(pattern, recursive=True))
    if not paths:
        raise FileNotFoundError(f"no shards matched {pattern}")
    samples = []
    for p in paths:
        with _opener(p)(p, "rb") as f:
            samples.extend(pickle.load(f))
    return samples


class BCDataset(Dataset):
    """Wraps raw {obs, action, outcome} samples; encodes lazily in __getitem__
    (cheap — pure python dict indexing, no torch ops until collate)."""

    def __init__(self, raw_samples):
        self.raw = raw_samples

    def __len__(self):
        return len(self.raw)

    def __getitem__(self, i):
        d = self.raw[i]
        obs, action, outcome = d["obs"], d["action"], d["outcome"]
        sel = obs.get("select")
        enc = encode_sample(obs, sel)
        label = action[0] if action else 0
        label = min(label, enc["n_actions"] - 1) if enc["n_actions"] else 0
        return enc, label, float(outcome or 0)


def collate(batch):
    B = len(batch)
    board_ids = torch.zeros(B, 13, dtype=torch.long)
    hand_ids = torch.zeros(B, 20, dtype=torch.long)
    discard_ids = torch.zeros(B, 20, dtype=torch.long)
    numeric = torch.zeros(B, 13, dtype=torch.float)
    action_type = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_card = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_attack = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_numeric = torch.zeros(B, MAX_ACTIONS, 4, dtype=torch.float)
    action_mask = torch.zeros(B, MAX_ACTIONS, dtype=torch.float)
    labels = torch.zeros(B, dtype=torch.long)
    values = torch.zeros(B, dtype=torch.float)

    for i, (enc, label, outcome) in enumerate(batch):
        board_ids[i] = torch.tensor(enc["board_ids"], dtype=torch.long)
        h = enc["hand_ids"][:20]
        hand_ids[i, :len(h)] = torch.tensor(h, dtype=torch.long)
        dcd = enc["discard_ids"][:20]
        discard_ids[i, :len(dcd)] = torch.tensor(dcd, dtype=torch.long)
        numeric[i] = torch.tensor(enc["numeric"], dtype=torch.float)
        n = enc["n_actions"]
        for j, a in enumerate(enc["actions"][:MAX_ACTIONS]):
            action_type[i, j] = a["type"]
            action_card[i, j] = a["card_id"]
            action_attack[i, j] = a["attack_id"]
            action_numeric[i, j] = torch.tensor(a["numeric"], dtype=torch.float)
            action_mask[i, j] = 1.0
        labels[i] = label
        values[i] = outcome

    return {
        "board_ids": board_ids, "hand_ids": hand_ids, "discard_ids": discard_ids,
        "numeric": numeric, "action_type": action_type, "action_card": action_card,
        "action_attack": action_attack, "action_numeric": action_numeric,
        "action_mask": action_mask, "labels": labels, "values": values,
    }


In [ ]:
%%writefile train_bc.py
"""BC training loop. Runs locally (CPU, small subset, for smoke-testing) or on
Kaggle (GPU, full data) — same code, just point --data at different globs.

Usage:
  python train_bc.py --data "../bc_data*.pkl.gz" --epochs 10 --out ptcg_bc.pth
"""
import argparse
import random
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from dataset import BCDataset, collate, load_shards
from model import PTCGNet


def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits, value = model(
                batch["board_ids"], batch["hand_ids"], batch["discard_ids"],
                batch["numeric"], batch["action_type"], batch["action_card"],
                batch["action_attack"], batch["action_numeric"], batch["action_mask"])
            pred = logits.argmax(dim=-1)
            correct += (pred == batch["labels"]).sum().item()
            total += pred.shape[0]
    model.train()
    return correct / max(total, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="../bc_data*.pkl.gz")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--out", default="ptcg_bc.pth")
    ap.add_argument("--limit", type=int, default=None, help="cap samples (smoke tests)")
    ap.add_argument("--val-frac", type=float, default=0.05)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")

    raw = load_shards(args.data)
    random.Random(0).shuffle(raw)
    if args.limit:
        raw = raw[: args.limit]
    print(f"loaded {len(raw)} samples")

    ds = BCDataset(raw)
    n_val = max(1, int(len(ds) * args.val_frac))
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(0))

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)

    model = PTCGNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    policy_loss_fn = nn.CrossEntropyLoss()
    value_loss_fn = nn.HuberLoss(delta=0.2)

    for epoch in range(args.epochs):
        total_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits, value = model(
                batch["board_ids"], batch["hand_ids"], batch["discard_ids"],
                batch["numeric"], batch["action_type"], batch["action_card"],
                batch["action_attack"], batch["action_numeric"], batch["action_mask"])
            p_loss = policy_loss_fn(logits, batch["labels"])
            v_loss = value_loss_fn(value, batch["values"])
            loss = p_loss + 0.5 * v_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        val_acc = evaluate(model, val_loader, device)
        print(f"epoch {epoch}: train_loss={total_loss/max(n_batches,1):.4f} "
              f"val_top1_acc={val_acc:.4f}")

    torch.save(model.state_dict(), args.out)
    print(f"saved {args.out}")


if __name__ == "__main__":
    main()


## Train

Gate from `docs/nn-training.md`: this notebook reports **held-out top-1 action-match accuracy** against v22 (a direct imitation-quality proxy). The competition-relevant gates — 65%+ vs random, ~50% vs v22 — require actually playing games with the trained net, which needs an agent wrapper (`training/README.md` next steps) run outside this notebook, e.g. locally via `training/ab_test.py`.

In [ ]:
!python train_bc.py --data "/kaggle/input/**/bc_data*.pkl" --epochs 10 --batch-size 256 --lr 1e-4 --out /kaggle/working/ptcg_bc_v1.pth


## Next steps

- Download `/kaggle/working/ptcg_bc_v1.pth`.
- Write a thin `agent(obs_dict)` wrapper around the checkpoint (load `PTCGNet`, encode the obs with `encode.py`, argmax the masked logits, map back to `select.option` indices) so it satisfies the same contract as `main.py` / `opponents/*.py`.
- Evaluate locally: `python training/ab_test.py <net_agent.py> main.py 400` (random-vs-net and v22-vs-net gates from `docs/nn-training.md`).
- If it clears ~50% vs v22, ship it to the ladder (single forward pass — no MCTS latency risk) and start self-play Phase 1.